# 02 · Stationäre Verteilungen und Zeitanteile

Die Aufgaben sind in einzelne Stationen eingeteilt. Lies jeweils zuerst die Aufgabenstellung und mach dir eigene Gedanken und probiere dann das jeweilige Widget aus. Speichere deine Codeänderungen mit **Strg + S**.

---

Wie bisher gilt: **Spalte = Start, Zeile = Ziel**. Verteilungen sind Spaltenvektoren; ein Schritt bedeutet $p^{(k+1)}=Mp^{(k)}$.

**Schwierigkeitsstufen**

| Stufe | Orientierung |
|---|---|
| Leicht | Ab Klasse 9: Wahrscheinlichkeiten, Baumdiagramme, Bruchrechnen. Oft genügen Ausprobieren und genaues Lesen. |
| Schwerer | Ab Klasse 10/11: Matrizen, Gleichungssysteme, mehrstufige Überlegungen und Argumentieren mit Begriffen aus dem Skript. |
| Schwer | Eher Klasse 12/13: Knobelaufgaben, anspruchsvollere Begründungen, Beweisideen und ungewohnte Modellierung. |

Die Klassenstufen sind eine Orientierung, keine Zugangsvoraussetzung. Du kannst schwierigere Teilaufgaben auch überspringen, die Reihenfolge ist nicht obligatorisch.

<br>

## Vorbereitung der Widgets
#### Klicke zuerst oben im Menü auf **Run → Run All Cells**. Den Code in der hier folgenden Zelle brauchst du nicht zu bearbeiten.

In [ ]:
# Pakete und Widgets vorbereiten.
import sys

try:
    import ipywidgets as W
except ModuleNotFoundError:
    if sys.platform == "emscripten":
        import piplite

        await piplite.install("ipywidgets>=8,<9")
        import ipywidgets as W
    else:
        raise RuntimeError("Bitte ipywidgets in dieser Python-Umgebung installieren lassen.")


import io
import html
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import display

PANELS = {}


C = np.array([[0.0, 0.0, 1.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])


MODELLE = {"Dreierzyklus": C, "Mit Wartepausen": (np.eye(3) + C) / 2, "Stillstand": np.eye(3)}


# Markowketten und gemeinsame Widget-Hilfen.
def entwicklung(M, p, n):
    result = [np.array(p, dtype=float)]
    for _ in range(n):
        result.append(M @ result[-1])
    return np.array(result)


def pfad_ziehen(M, start, n, seed):
    rng = np.random.default_rng(seed)
    grenzen = np.cumsum(M, axis=0)
    grenzen[-1, :] = 1
    uniforms = rng.random(n)
    x = np.empty(n + 1, dtype=int)
    x[0] = start
    for t, u in enumerate(uniforms, start=1):
        x[t] = np.searchsorted(grenzen[:, x[t - 1]], u, side="right")
    return x


def regler(value, maximum=100, minimum=0):
    return W.IntSlider(
        value=value,
        min=minimum,
        max=maximum,
        continuous_update=True,
        layout=W.Layout(width="480px", max_width="100%"),
    )


def feld(text, w):
    return W.VBox([W.HTML(html.escape(text)), w])


def bild():
    return W.Image(format="png", layout=W.Layout(width="700px", max_width="100%"))


def zeichnen(fig, w):
    with io.BytesIO() as f:
        fig.savefig(f, format="png", dpi=105)
        w.value = f.getvalue()


def tabelle(head, rows):
    h = "".join(f'<th style="padding:4px 10px">{html.escape(str(x))}</th>' for x in head)
    body = "".join(
        "<tr>"
        + "".join(
            f'<td style="padding:4px 10px;text-align:center">{html.escape(str(x))}</td>'
            for x in row
        )
        + "</tr>"
        for row in rows
    )
    return '<div style="overflow-x:auto"><table><tr>' + h + "</tr>" + body + "</table></div>"


def matrixtext(M):
    return tabelle(
        ["Ziel ↓ / Start →"] + list(range(1, len(M) + 1)),
        [[i + 1] + [f"{v:.4f}" for v in row] for i, row in enumerate(M)],
    )


def zahlenfelder(n):
    return [
        W.Text(
            placeholder="Deine Zahl / dein Bruch",
            layout=W.Layout(width="200px", max_width="100%"),
        )
        for _ in range(n)
    ]


def verteilung(ws):
    try:
        q = np.array([float(Fraction(w.value.strip().replace(",", "."))) for w in ws])
    except (ValueError, ZeroDivisionError, OverflowError):
        raise ValueError("Bitte alle Zahlenfelder ausfüllen; Brüche sind erlaubt.")
    if not np.isfinite(q).all() or np.any(q < 0) or np.any(q > 1):
        raise ValueError("Alle Einträge müssen zwischen 0 und 1 liegen.")
    if not np.isclose(q.sum(), 1, atol=1e-6, rtol=0):
        raise ValueError("Die Einträge müssen sich zu 1 summieren.")
    return q / q.sum()


# Stationen zu Verteilungen und Zeitanteilen.
def station_stationaer():
    model = W.Dropdown(options=["Wettermodell", "Drei Zustände (Zusatz)"])
    matrices = [
        np.array([[0.8, 0.3], [0.2, 0.7]]),
        np.array([[0.1, 0.2, 0.3], [0.4, 0.5, 0.6], [0.5, 0.3, 0.1]]),
    ]
    ws = zahlenfelder(3)
    inputs = [feld(f"q_{i+1}", w) for i, w in enumerate(ws)]
    matrix, info = W.HTML(), W.HTML()
    btn = W.Button(
        description="Meinen Kandidaten prüfen",
        button_style="info",
        layout=W.Layout(width="240px"),
    )
    r = {"model": model, "fields": ws, "button": btn, "last": None, "info": info}

    def invalidate(_=None):
        r["last"] = None
        info.value = "Gib deinen Kandidaten ein und prüfe ihn."

    def changed(_=None):
        idx = 0 if model.value == "Wettermodell" else 1
        r["M"] = matrices[idx]
        matrix.value = matrixtext(r["M"])
        inputs[2].layout.display = "none" if idx == 0 else ""
        for w in ws:
            w.value = ""
        invalidate()

    def check(_):
        r["last"] = None
        try:
            q = verteilung(ws[: len(r["M"])])
        except ValueError as e:
            info.value = html.escape(str(e))
            return
        Mq = r["M"] @ q
        ok = bool(np.allclose(Mq, q, atol=1e-6, rtol=0))
        info.value = tabelle(
            ["Zustand", "Dein q", "Mq", "Mq − q"],
            [
                [i + 1, f"{x:.5f}", f"{y:.5f}", f"{y-x:+.5f}"]
                for i, (x, y) in enumerate(zip(q, Mq))
            ],
        ) + (
            "<b>Stationär bis auf Rundung.</b>"
            if ok
            else "Noch nicht stationär: Prüfe die Gleichungen und die Normalisierung."
        )
        r["last"] = {"correct": ok, "q": q, "next": Mq}

    for w in ws:
        w.observe(invalidate, names="value")
    model.observe(changed, names="value")
    btn.on_click(check)
    display(W.VBox([model, matrix] + inputs + [btn, info]))
    changed()
    PANELS["stationaer"] = r
    return r


def station_verteilungen():
    model = W.Dropdown(options=list(MODELLE))

    # Integer selection values avoid browser-side rounding of 1/3 to 0.33.
    # Each unit is 1/300, so ordinary percentages and thirds are all selectable.
    def label(u):
        if u == 100:
            return "1/3"
        if u == 200:
            return "2/3"
        return f"{u/300:.3f}"

    def options(maximum):
        return [(label(u), u) for u in range(maximum + 1)]

    def probability_slider(u, maximum):
        return W.SelectionSlider(
            options=options(maximum),
            value=u,
            continuous_update=True,
            layout=W.Layout(width="480px", max_width="100%"),
        )

    p1 = probability_slider(180, 300)
    p2 = probability_slider(90, 120)
    equal = W.Button(
        description="Gleichverteilung",
        layout=W.Layout(width="250px"),
        tooltip="Setzt p₁ und p₂ auf 1/3; danach direkt an den Reglern weiterschieben.",
    )
    inputs = W.VBox(
        [feld("Anfangswahrscheinlichkeit p₁", p1), feld("Anfangswahrscheinlichkeit p₂", p2)]
    )
    n = regler(12, 60, 1)
    pic = bild()
    pic.layout.width = "850px"
    info = W.HTML()
    initial = W.HTML()
    fig, axs = plt.subplots(1, 3, figsize=(8, 2.8), sharey=True)
    lines = []
    for i, ax in enumerate(axs):
        lines.append(ax.plot([], [], "o-", ms=3, color=["#2875aa", "#ad6500", "#278353"][i])[0])
        ax.set(ylim=(-0.04, 1.04), xlabel="Schritt k", title=f"Zustand {i+1}")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=4))
        ax.grid(alpha=0.2)
    axs[0].set_ylabel("Wahrscheinlichkeit")
    fig.tight_layout()
    plt.close(fig)
    r = {
        "model": model,
        "p1": p1,
        "p2": p2,
        "equal": equal,
        "n": n,
        "bild": pic,
        "fig": fig,
        "lock": False,
        "last": None,
    }

    def update(_=None):
        if r["lock"]:
            return
        maximum = 300 - p1.value
        if len(p2.options) != maximum + 1:
            chosen = min(p2.value, maximum)
            r["lock"] = True
            try:
                p2.options = options(maximum)
                p2.value = chosen
            finally:
                r["lock"] = False
        units = np.array([p1.value, p2.value, 300 - p1.value - p2.value])
        p0 = units / 300
        uniform = bool(np.all(units == 100))
        # Analytically stationary for all three models: reuse the identical value
        # at every step instead of repeatedly introducing floating-point roundoff.
        ps = (
            np.tile(p0, (n.value + 1, 1))
            if uniform
            else entwicklung(MODELLE[model.value], p0, n.value)
        )
        initial.value = (
            "<b>Gleichverteilung: (1/3, 1/3, 1/3).</b>"
            if uniform
            else "Startverteilung: ("
            + ", ".join(f"{x:.4f}" for x in p0)
            + "). p₃ ist der Rest auf 1; p₂ wird bei Bedarf begrenzt."
        )
        for i, (line, ax) in enumerate(zip(lines, axs)):
            line.set_data(np.arange(n.value + 1), ps[:, i])
            ax.set_xlim(0, n.value)
        zeichnen(fig, pic)
        info.value = (
            "Verteilung nach " + str(n.value) + " Schritten: (1/3, 1/3, 1/3)."
            if uniform
            else "Verteilung nach "
            + str(n.value)
            + " Schritten: ("
            + ", ".join(f"{x:.4f}" for x in ps[-1])
            + ")."
        )
        r["last"] = {"p0": p0, "distributions": ps, "uniform": uniform}

    def uniform_start(_):
        r["lock"] = True
        try:
            p1.value = 100
            p2.options = options(200)
            p2.value = 100
        finally:
            r["lock"] = False
        update()

    for w in (model, p1, p2, n):
        w.observe(update, names="value")
    equal.on_click(uniform_start)
    display(W.VBox([model, equal, inputs, initial, feld("Anzahl Schritte", n), pic, info]))
    update()
    PANELS["verteilungen"] = r
    return r


def station_zeitanteile():
    model = W.Dropdown(options=list(MODELLE))
    start = W.Dropdown(
        options=[
            ("Sicher Zustand 1", 0),
            ("Sicher Zustand 2", 1),
            ("Sicher Zustand 3", 2),
            ("Gleichverteilt auslosen", -1),
        ]
    )
    n = W.SelectionSlider(
        options=[30, 31, 32, 100, 300, 1000, 3000, 10000, 30000],
        value=30,
        continuous_update=True,
        layout=W.Layout(width="480px", max_width="100%"),
    )
    btn = W.Button(
        description="Neuen Versuch auslosen",
        button_style="info",
        layout=W.Layout(width="230px"),
    )
    pic = bild()
    info = W.HTML()
    fig, axs = plt.subplots(2, 1, figsize=(6.7, 4.7))
    (line1,) = axs[0].plot([], [], "o-", ms=3, color="#2875aa")
    (line2,) = axs[1].plot([], [], color="#ad6500")
    axs[0].set(
        xlabel="Schritt k (Ausschnitt)", ylabel="P(Zustand 1)", ylim=(-0.04, 1.04), xlim=(0, 30)
    )
    axs[1].set(
        xlabel="Gezählte Schritte n (logarithmisch)",
        ylabel="Zeitanteil in Zustand 1",
        xscale="log",
        xlim=(1, 300),
        ylim=(-0.04, 1.04),
    )
    for ax in axs:
        ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.close(fig)
    r = {
        "model": model,
        "start": start,
        "n": n,
        "button": btn,
        "seed": 2026,
        "key": None,
        "bild": pic,
        "last": None,
    }

    def update(_=None):
        key = (model.value, start.value, r["seed"])
        M = MODELLE[model.value]
        p0 = np.ones(3) / 3 if start.value == -1 else np.eye(3)[:, start.value]
        if r["key"] != key:
            actual = int(np.random.default_rng(r["seed"]).choice(3, p=p0))
            r["path"] = pfad_ziehen(M, actual, 30000, r["seed"] + 1000000)
            r["sums"] = np.cumsum(r["path"][1:] == 0)
            r["key"] = key
        N = n.value
        idx = np.unique(np.geomspace(1, N, min(N, 600)).astype(int))
        ps = entwicklung(M, p0, 30)
        line1.set_data(np.arange(31), ps[:, 0])
        line2.set_data(idx, r["sums"][idx - 1] / idx)
        axs[1].set_xlim(1, N)
        zeichnen(fig, pic)
        counts = np.bincount(r["path"][1 : N + 1], minlength=3)
        freq = counts / N
        info.value = (
            f'Tatsächlich ausgeloster Start: <b>Zustand {r["path"][0]+1}</b>. Oben: Verteilung vor Kenntnis dieses Starts.'
            + tabelle(
                ["Zustand", "Besuche 1 bis N", "Zeitanteil"],
                [[i + 1, int(counts[i]), f"{freq[i]:.5f}"] for i in range(3)],
            )
        )
        r["last"] = {
            "path": r["path"][: N + 1].copy(),
            "counts": counts,
            "frequencies": freq,
            "p0": p0,
            "distributions": ps,
        }

    def new(_):
        r["seed"] += 1
        update()

    for w in (model, start, n):
        w.observe(update, names="value")
    btn.on_click(new)
    display(W.VBox([model, start, feld("Gezählte Schritte N", n), btn, pic, info]))
    update()
    PANELS["zeit"] = r
    return r


# Matrixeingaben und Stationen zu Zyklus und Glücksspiel.
def matrixfelder(labels):
    size = len(labels)
    cells = [
        [W.Text(placeholder="…", layout=W.Layout(width="60px")) for _ in labels] for _ in labels
    ]
    widgets = [W.HTML("<small>Ziel ↓<br>Start →</small>")] + [
        W.HTML(f"<b>{s}</b>") for s in labels
    ]
    for label, row in zip(labels, cells):
        widgets += [W.HTML(f"<b>{label}</b>")] + row
    grid = W.GridBox(
        widgets,
        layout=W.Layout(
            grid_template_columns=f"repeat({size+1}, 65px)",
            grid_gap="4px",
            overflow="auto",
            max_width="100%",
        ),
    )
    return grid, cells


def matrix_lesen(cells):
    try:
        a = np.array(
            [[float(Fraction(w.value.strip().replace(",", "."))) for w in row] for row in cells]
        )
    except (ValueError, ZeroDivisionError, OverflowError):
        raise ValueError("Bitte alle Felder mit Zahlen oder Brüchen ausfüllen.")
    if not np.isfinite(a).all() or np.any(a < 0) or np.any(a > 1):
        raise ValueError("Die Einträge müssen zwischen 0 und 1 liegen.")
    return a


def station_zyklus():
    k = W.Dropdown(
        options=[2, 3, 4, 5, 300000, 300001, 300002], value=2, description="Exponent k:"
    )
    grid, cells = matrixfelder([1, 2, 3])
    button = W.Button(
        description="Meine Matrix prüfen", button_style="info", layout=W.Layout(width="230px")
    )
    feedback = W.HTML("Trage deine Matrix ein. Die Zelle gibt keine Musterlösung vor.")
    r = {"k": k, "cells": cells, "button": button, "feedback": feedback, "last": None}

    def invalidate(_):
        r["last"] = None
        feedback.value = "Eingabe geändert; erneut prüfen."

    def check(_):
        r["last"] = None
        try:
            A = matrix_lesen(cells)
        except ValueError as e:
            feedback.value = html.escape(str(e))
            return
        if not np.allclose(A.sum(axis=0), 1, atol=1e-8, rtol=0):
            feedback.value = "Prüfe die Spaltensummen: Jede muss 1 sein."
            return
        C = np.array([[0, 0, 1], [1, 0, 0], [0, 1, 0]])
        stimmen = np.all(
            np.isclose(A, np.linalg.matrix_power(C, k.value), atol=1e-8, rtol=0), axis=0
        )
        ok = bool(stimmen.all())
        feedback.value = (
            f"<b>Richtig für k = {k.value}.</b> Begründe dein Ergebnis mit den Übergängen."
            if ok
            else f"{stimmen.sum()} von 3 Spalten stimmen. Verfolge für jede Spalte ihren Startzustand."
        )
        r["last"] = {"correct": ok, "candidate": A}

    for row in cells:
        for w in row:
            w.observe(invalidate, names="value")
    k.observe(invalidate, names="value")
    button.on_click(check)
    display(W.VBox([k, grid, button, feedback]))
    PANELS["zyklus"] = r
    return r


def station_gluecksspiel():
    labels = list(range(-3, 4))
    grid, cells = matrixfelder(labels)
    nullen = W.Button(description="Leere Felder mit 0 füllen", layout=W.Layout(width="230px"))
    checkbutton = W.Button(
        description="Meine Matrix prüfen", button_style="info", layout=W.Layout(width="230px")
    )
    feedback = W.HTML(
        "Baue zuerst die Übergangsmatrix aus den Spielregeln oder arbeite mit der vorgegebenen Matrix weiter."
    )
    vorgegeben = W.Button(
        description="Mit vorgegebener Matrix weiterarbeiten",
        layout=W.Layout(width="340px", max_width="100%"),
    )
    k = regler(2, 100, 0)
    k.disabled = True
    start = W.Dropdown(
        options=[(f"{s} €", s) for s in labels] + [("Gleichverteilt", "gleich")],
        value=0,
        disabled=True,
    )
    bild = W.Image(format="png", layout=W.Layout(width="700px", max_width="100%"))
    info, powertext = W.HTML(), W.HTML()
    powerbox = W.Accordion(children=[powertext], selected_index=None)
    powerbox.set_title(0, "M^k für alle Startzustände (aufklappen)")
    result = W.VBox(
        [feld("Anzahl Schritte k", k), feld("Anfangsverteilung", start), bild, info, powerbox],
        layout=W.Layout(display="none"),
    )
    fig, ax = plt.subplots(figsize=(6.7, 3.0))
    bars = ax.bar(labels, np.zeros(7), color="#2875aa")
    ax.set(
        xticks=labels,
        xlabel="Gewinn / Verlust in Euro",
        ylabel="Wahrscheinlichkeit",
        ylim=(0, 1.05),
    )
    fig.tight_layout()
    plt.close(fig)
    r = {
        "cells": cells,
        "button": checkbutton,
        "zeros": nullen,
        "k": k,
        "start": start,
        "feedback": feedback,
        "bild": bild,
        "matrix": None,
        "last": None,
        "result": result,
        "vorgegeben": vorgegeben,
    }

    def invalidate(_=None):
        r["matrix"] = None
        r["last"] = None
        k.disabled = True
        start.disabled = True
        result.layout.display = "none"
        bild.value = b""
        powertext.value = ""
        info.value = ""
        feedback.value = "Matrix geändert; bitte erneut prüfen."

    def update(_=None):
        if r["matrix"] is None:
            return
        p0 = np.ones(7) / 7 if start.value == "gleich" else np.eye(7)[:, start.value + 3]
        power = np.linalg.matrix_power(r["matrix"], k.value)
        p = power @ p0
        for bar, v in zip(bars, p):
            bar.set_height(v)
        zeichnen(fig, bild)
        info.value = (
            f"Nach {k.value} Schritten: P(−3 €) = <b>{p[0]:.4f}</b>, P(+3 €) = <b>{p[-1]:.4f}</b>; "
            f"noch im Spiel: <b>{p[1:-1].sum():.4f}</b>."
        )
        powertext.value = tabelle(
            ["Ziel ↓ / Start →"] + labels,
            [[s] + [f"{v:.4f}" for v in row] for s, row in zip(labels, power)],
        )
        r["last"] = {"distribution": p, "power": power, "p0": p0}

    def check(_):
        invalidate()
        try:
            A = matrix_lesen(cells)
        except ValueError as e:
            feedback.value = html.escape(str(e))
            return
        if not np.allclose(A.sum(axis=0), 1, atol=1e-8, rtol=0):
            feedback.value = "Jede Spalte muss sich zu 1 summieren."
            return
        expected = np.zeros((7, 7))
        expected[0, 0] = expected[-1, -1] = 1
        for j in range(1, 6):
            expected[j - 1, j] = expected[j + 1, j] = 0.5
        wrong = [
            str(labels[j])
            for j in range(7)
            if not np.allclose(A[:, j], expected[:, j], atol=1e-8, rtol=0)
        ]
        if wrong:
            feedback.value = (
                "Prüfe die Spalten für Start "
                + ", ".join(wrong)
                + " €. An den Rändern bleibt der Zustand unverändert."
            )
            return
        r["matrix"] = A.copy()
        k.disabled = False
        start.disabled = False
        result.layout.display = ""
        feedback.value = "<b>Die Matrix stimmt.</b> Die Regler darunter aktualisieren jetzt die Verteilung direkt."
        update()

    def use_given(_):
        A = np.zeros((7, 7))
        A[0, 0] = A[-1, -1] = 1
        for j in range(1, 6):
            A[j - 1, j] = A[j + 1, j] = 0.5
        r["matrix"] = A
        k.disabled = False
        start.disabled = False
        result.layout.display = ""
        feedback.value = "Vorgegebene Matrix aktiv. Deine Eingaben oben bleiben unverändert."
        update()

    vorgegeben.on_click(use_given)

    def fill(_):
        for row in cells:
            for w in row:
                if not w.value.strip():
                    w.value = "0"

    for row in cells:
        for w in row:
            w.observe(invalidate, names="value")
    for w in (k, start):
        w.observe(update, names="value")
    nullen.on_click(fill)
    checkbutton.on_click(check)
    display(W.VBox([grid, nullen, checkbutton, vorgegeben, feedback, result]))
    PANELS["spiel"] = r
    r["update"] = update
    return r


print("Bereit – weiter mit Station 1.")


<br><br>

## 1 · Ein Dreierzyklus (Skriptaufgabe 1.35)

Gegeben ist

$$C=\begin{pmatrix}0&0&1\\1&0&0\\0&1&0\end{pmatrix}.$$

Die Spalte bezeichnet den Startzustand, die Zeile das Ziel; die Zustände heißen 1, 2 und 3.

1. **Zeichne die Übergänge auf Papier und bestimme $C^2$.** Trage dein Ergebnis in das leere Gitter ein. Die Schaltfläche prüft deine Matrix, ohne eine Musterlösung einzutragen. <span style="color: #707070;">(schwerer)</span>
2. Untersuche ebenso $C^3$, $C^4$ und $C^5$. <span style="color: #707070;">(schwerer)</span>
3. Welche Regel vermutest du für $C^k$? Begründe damit deine Ergebnisse für **k = 300 000, 300 001 und 300 002**, bevor du sie prüfst. <span style="color: #707070;">(schwerer)</span>


In [ ]:
station_1 = station_zyklus()

<br><br>

## 2 · Stationäre Verteilungen

Eine **stationäre Verteilung** erfüllt $Mq=q$, $q_i\geq0$ und $\sum_iq_i=1$.

1. **Berechne die stationäre Verteilung q für das Wettermodell** mit a = 0,8 und b = 0,3 auf Papier. <span style="color: #707070;">(schwerer)</span>
2. Trage deinen Kandidaten für q ein (Zustand 1 = Sonne, 2 = Regen). Die Zelle zeigt **dein q, Mq und die Differenz**; sie gibt keine Lösung vor. <span style="color: #707070;">(leicht)</span>
3. **Zusatz:** Wähle „Drei Zustände“ und berechne die stationäre Verteilung q auch für dieses Modell. Trage deinen Kandidaten für q ein und prüfe ihn. <span style="color: #707070;">(schwerer)</span>


In [ ]:
station_2 = station_stationaer()

<br><br>

## 3 · Stationär – aber auch ein Grenzwert?

| Modell | Übergänge |
|---|---|
| Dreierzyklus C | 1 → 2 → 3 → 1 |
| Mit Wartepausen: (I + C)/2 | Mit Wahrscheinlichkeit 1/2 bleiben, sonst im Zyklus weitergehen |
| Stillstand: I | Immer im aktuellen Zustand bleiben |

Die Regler wählen die Anfangswahrscheinlichkeiten $p_1$ und $p_2$ für Zustand 1 und 2; die Wahrscheinlichkeit $p_3=1-p_1-p_2$ wird automatisch ergänzt. 

1. Stelle **p = (0,6; 0,3; 0,1)** ein. Warum konvergieren die Verteilungen im Dreierzyklus nicht, sobald man aber eine Wartewahrscheinlichkeit von 1/2 einstellt ("Mit Wartepausen") schon? <span style="color: #707070;">(schwerer)</span>
2. Klicke auf **„Gleichverteilung“**. Der Button setzt die sichtbaren Regler auf $p_1=p_2=1/3$, sodass auch $p_3=1/3$ ist. Danach kannst du direkt an den Reglern weiterschieben. Warum bewegt sie sich in keinem der drei Modelle? Reicht ein stationäres q also aus, damit sich jede Anfangsverteilung ihm annähert? <span style="color: #707070;">(schwerer)</span>
3. Wie viele verschiedene stationäre Verteilung gibt es beim "Stillstand"? <span style="color: #707070;">(schwerer)</span>
4. Ordne die Modelle nach **Irreduzibilität und Periode** ein. <span style="color: #707070;">(schwerer)</span>


In [ ]:
station_3 = station_verteilungen()

<br><br>

## 4 · Stationarität und Zeitanteile

In der Grafik steht oben $P(X_k=1)$, unten der Zeitanteil in Zustand 1 unter den Schritten **1 bis n**. Der Startbesuch zählt nicht mit. Bei „Gleichverteilt auslosen“ wird zu Beginn jedes Versuchs genau ein Startzustand gezogen; die obere Kurve berücksichtigt die Verteilung dieses Zufallsstarts.

1. **„Ohne Konvergenz der Verteilungen können auch die Zeitanteile nicht konvergieren.“** Widerlege die Behauptung mit dem Dreierzyklus ab Zustand 1. Berechne zunächst die Besuchszahlen aller drei Zustände für **N = 30, 31 und 32** und prüfe sie dann. Begründe, warum trotzdem jeder Zeitanteil gegen 1/3 konvergiert, indem du $N=3m+r$ betrachtest. <span style="color: #707070;">(schwerer)</span>
2. **„Bei stationärem Start stimmen die langfristigen Zeitanteile eines einzelnen Pfads mit der Startverteilung überein.“** Belege oder widerlege diese Behauptung mithilfe eines Beispiels in der Grafik. <span style="color: #707070;">(leicht)</span>
3. Welche Rolle spielt hier die **Irreduzibilität**? Prüfe zum Vergleich das Modell mit Wartepausen. <span style="color: #707070;">(schwerer)</span>


In [ ]:
station_4 = station_zeitanteile()

<br><br>

## 5 · Ein Glücksspiel (Skriptaufgabe 1.36)

Die Zustände $-3,-2,-1,0,1,2,3$ geben den Gewinn bzw. Verlust in Euro an. Pro Schritt gewinnt oder verliert man mit Wahrscheinlichkeit $1/2$ einen Euro. Bei **−3 oder +3 endet das Spiel**; danach bleibt der Zustand unverändert.

1. **Baue die Übergangsmatrix M selbst, spaltenweise.** Trage zunächst die positiven Einträge ein und fülle die übrigen Felder über die Nullenschaltfläche. Eine korrekt geprüfte Matrix öffnet das Experiment. Du kannst diese Teilaufgabe auch überspringen: **„Mit vorgegebener Matrix weiterarbeiten“** öffnet direkt die Untersuchung in Teilaufgabe 2 und 3. <span style="color: #707070;">(schwerer)</span>
2. Vergleiche **k = 2, 10, 50, 100** und verschiedene Starts. Die Balken zeigen $M^kp^{(0)}$; die vollständige Matrix $M^k$ kannst du aufklappen. <span style="color: #707070;">(leicht)</span>
3. Was vermutest du für sehr große k und wie hängt es vom Start ab? Versuche deine Vermutung zu beweisen. Probiere auch eine gleichverteilte Anfangsverteilung. <span style="color: #707070;">(schwerer)</span>


In [ ]:
station_5 = station_gluecksspiel()

<br><br>

## Kleine Codeaufgabe · Vom Spitzenrating zur Zahlungsunfähigkeit (Skriptaufgabe 1.33)

Die Matrix unten beschreibt die jährlichen Übergänge zwischen **AAA, AA, A, BBB, BB, B, CCC/CC/C und D** aus dem Skript. D bedeutet Zahlungsunfähigkeit. Spalten sind Startratings, Zeilen sind Zielratings.

1. Ein Unternehmen startet mit **AAA**. Wie wahrscheinlich ist es, dass es **zehn Jahre später zahlungsunfähig** ist? Notiere zunächst, welchen Eintrag welcher Matrix du dafür brauchst. <span style="color: #707070;">(schwerer)</span>
2. Ersetze im Code `None` durch die passende Matrixpotenz. `np.linalg.matrix_power(A, k)` berechnet $A^k$; `A ** k` potenziert dagegen jeden Eintrag einzeln. Die Ausgabe zeigt anschließend die Verteilung nach zehn Jahren. Gib die gesuchte Wahrscheinlichkeit auch in Prozent an. <span style="color: #707070;">(schwerer)</span>
3. Warum bedeutet „nach zehn Jahren in D“ hier zugleich „innerhalb der ersten zehn Jahre zahlungsunfähig geworden“? Begründe dies anhand der Spalte für D. <span style="color: #707070;">(schwerer)</span>


In [ ]:
# Übergangsmatrix aus Aufgabe 1.33: ein Schritt entspricht einem Jahr.
ratings = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC/CC/C", "D"]
M_rating = np.array([
    [.9081, .0057, .0004, .0001, .0002, .0000, .0000, .0000],
    [.0833, .9051, .0205, .0014, .0004, .0004, .0000, .0000],
    [.0068, .0810, .9128, .0384, .0018, .0013, .0020, .0000],
    [.0006, .0058, .0582, .9032, .0546, .0023, .0032, .0000],
    [.0008, .0006, .0053, .0442, .8346, .0585, .0096, .0000],
    [.0002, .0011, .0016, .0077, .0820, .8290, .1224, .0000],
    [.0000, .0002, .0003, .0016, .0086, .0489, .5815, .0000],
    [.0002, .0005, .0009, .0034, .0178, .0596, .2813, 1.0000],
])

M_zehn = None  # Hier deine Rechnung für die Zehn-Jahres-Übergangsmatrix ergänzen.

if M_zehn is None:
    print("Ergänze zuerst die Matrixpotenz bei M_zehn und führe die Zelle erneut aus.")
else:
    M_zehn = np.asarray(M_zehn, dtype=float)
    if (M_zehn.shape != (8, 8) or not np.isfinite(M_zehn).all()
            or np.any(M_zehn < 0) or not np.allclose(M_zehn.sum(axis=0), 1)):
        print("Prüfe deine Rechnung: Das Ergebnis muss eine Übergangsmatrix sein.")
    else:
        p_start = np.array([1., 0., 0., 0., 0., 0., 0., 0.])  # Start: AAA
        p_zehn = M_zehn @ p_start
        for rating, anteil in zip(ratings, p_zehn):
            print(f"{rating:8s}: {anteil:.8f} = {100*anteil:.5f} %")
